# Mpox classifier generalization - Kaggle run

**Before running**
1. Settings → Accelerator: **GPU** · Internet: **On**
2. Add Input → search and add both datasets:
   - `joydippaul/mpox-skin-lesion-dataset-version-20-msld-v20` (MSLD v2.0)
   - `dipuiucse/monkeypoxskinimagedataset` (MSID)
3. `REPO_URL` below already points to the project repository.

Run cells top to bottom. Run the **smoke test** before the full study.

In [ ]:
REPO_URL = "https://github.com/MirRahibHussain/mpox-generalization.git"  # your repository

import os
!rm -rf /kaggle/working/mpox-generalization
!git clone -q {REPO_URL} /kaggle/working/mpox-generalization
%cd /kaggle/working/mpox-generalization
!pip install -q grad-cam
import torch, timm
print("torch", torch.__version__, "| timm", timm.__version__, "| GPU:", torch.cuda.is_available())

## Find the two datasets

In [ ]:
def find_dirs(*keywords, top="/kaggle/input", max_depth=4):
    hits = []
    for root, dirs, _ in os.walk(top):
        depth = root[len(top):].count(os.sep)
        if depth >= max_depth:
            dirs[:] = []
        if any(k in os.path.basename(root).lower() for k in keywords):
            hits.append(root)
    return sorted(hits, key=len)

msld = find_dirs("msld", "mpox-skin-lesion")
msid = find_dirs("monkeypoxskinimage", "msid", "monkeypox-skin-image")
print("MSLD candidates:", msld[:3])
print("MSID candidates:", msid[:3])
MSLD, MSID = msld[0], msid[0]   # override by hand if the guess is wrong

## Step 1 - index
Check that MSLD v2.0 shows **755 unique images** and fold lines (fold 1 ... fold 5).

In [ ]:
!python scripts/01_build_index.py --data "msld_v2={MSLD}" --data "msid={MSID}" --out data/index

## Step 2 - embedding near-duplicates (GPU)
Look at the review sheet. For true duplicates, set `accept=1` in the next cell.

In [ ]:
!python scripts/02_embedding_pairs.py --index data/index
from IPython.display import Image as Show, display
if os.path.exists("data/index/embedding_review_sheet.png"):
    display(Show("data/index/embedding_review_sheet.png"))

In [ ]:
import pandas as pd
pairs = pd.read_csv("data/index/embedding_pairs.csv")
# Pairs are numbered #0, #1, ... on the sheet in the same order as the rows with accept == 0.
# Example: confirm sheet pairs #0 and #3 as duplicates:
# review_rows = pairs.index[pairs.accept == 0]
# pairs.loc[review_rows[[0, 3]], "accept"] = 1
pairs.to_csv("data/index/embedding_pairs.csv", index=False)
print(pairs.accept.value_counts())

## Step 3 - duplicate audit and skin tone (ITA)

In [ ]:
!python scripts/03_duplicate_audit.py --index data/index
display(Show("data/index/cross_dataset_duplicates.png")) if os.path.exists("data/index/cross_dataset_duplicates.png") else print("no cross-dataset duplicates found")

## Step 4 - splits and leakage check

In [ ]:
!python scripts/04_make_splits.py --index data/index --out splits

## Smoke test (about 5 minutes)
One epoch, one model, one split per protocol. If this fails, stop and fix before the full run.

In [ ]:
!python scripts/05_run_experiments.py --smoke

## Full study - binary task

Resumable: if the session stops, re-run this cell and finished runs are skipped.
For a long run use **Save Version → Save & Run All** so it runs in the background.

In [ ]:
!python scripts/05_run_experiments.py --tasks binary --save-ckpt

## Optional - 4-class task (shared classes)

In [ ]:
# !python scripts/05_run_experiments.py --tasks 4class

## Step 6 - tables and figure

In [ ]:
!python scripts/06_summarize.py --results results --index data/index
display(Show("results/fig_protocols.png"))

## Step 7 - Grad-CAM sheets
Open `gradcam_review.csv` in each folder and fill `attention` by eye: lesion / background / mixed.

In [ ]:
import glob
for run in sorted(glob.glob("results/binary_P3_*_rep0/resnet50")) + sorted(glob.glob("results/binary_P3_*_rep0/convnext_tiny")):
    !python scripts/07_gradcam.py --run {run} --index data/index

## Download everything

In [ ]:
!zip -qr /kaggle/working/outputs.zip data/index splits results -x "*.pt" "*.npy"
print("Download /kaggle/working/outputs.zip from the Output panel")